# Approval Check — Epic on FHIR Model

Deployment job **approval task** (task name must start with 'approval').

Checks Unity Catalog tags for manual approval before promoting the model to
"champion" and updating the serving endpoint.

**Approval pattern**: Tag `deployment.approval` set to `approved` on the model
version (or model-wide tag inherits to all versions).

**Required job parameters**: `model_name`, `model_version`

This task **fails** if approval tag is missing or not 'approved', which blocks
the downstream deployment task.

In [0]:
dbutils.widgets.text("model_name", "", "Model Name (catalog.schema.model)")
dbutils.widgets.text("model_version", "", "Model Version")

model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")

assert model_name, "model_name parameter is required"
assert model_version, "model_version parameter is required"

print(f"Checking approval for {model_name} v{model_version}")

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

try:
    model_version_details = client.get_model_version(model_name, model_version)
    tags = model_version_details.tags or {}
    
    # Check for deployment.approval tag (set via UC UI or API)
    approval_tag = tags.get("deployment.approval", "")
    
    if approval_tag.lower() == "approved":
        print(f"✓ Model version {model_version} is approved for deployment")
        print(f"  Tag: deployment.approval = {approval_tag}")
    else:
        raise ValueError(
            f"Model version {model_version} is not approved for deployment.\n"
            f"  Current tag value: deployment.approval = '{approval_tag}'\n"
            f"  Expected: 'approved'\n\n"
            f"To approve, set the UC tag on this model version via:\n"
            f"  1. Unity Catalog UI: Browse to {model_name} v{model_version} > Tags\n"
            f"  2. API: client.set_model_version_tag('{model_name}', {model_version}, 'deployment.approval', 'approved')\n"
        )
        
except Exception as e:
    print(f"✗ Approval check failed: {e}")
    raise

In [0]:
import json

dbutils.notebook.exit(json.dumps({
    "model_name": model_name,
    "model_version": model_version,
    "approval": "passed",
    "approval_tag": approval_tag,
}))